## Building a GPT

Companion notebook to the [Zero To Hero](https://karpathy.ai/zero-to-hero.html) video on GPT.

In [2]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-11-12 06:32:17--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.006s  

2026-11-12 06:32:18 (179 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [5]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string


In [6]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [7]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [10]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #torch.randint的語法,前者是最大值 至少抽出來之後能往後走block_size個；
    #(batch_size，)就是輸出的數據長什麽樣子啦 抽出來batch_size組扔進去
    x = torch.stack([data[i:i+block_size] for i in ix])
    #去data這個list[int]裏面把i到i+blocksize-1這些都拿出來
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    #去data這個list[int]裏面把i+1到i+blocksize這些都拿出來

    #stack就是曡羅漢 把抓出來的batch_size個長度爲block_size的tensor堆叠
    #變成[B,T]
    return x, y

xb, yb = get_batch('train') #[B,T]
print('inputs:') #[B,T]
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [11]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [13]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [14]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


4.65630578994751


In [15]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


oTo.JUZ!!zqe!
xBP qbs$Gy'AcOmrLwwt
p$x;Seh-onQbfM?OjKbn'NwUAW -Np3fkz$FVwAUEa-wzWC -wQo-R!v -Mj?,SPiTyZ;o-opr$mOiPJEYD-CfigkzD3p3?zvS;ADz;.y?o,ivCuC'zqHxcVT cHA
rT'Fd,SBMZyOslg!NXeF$sBe,juUzLq?w-wzP-h
ERjjxlgJzPbHxf$ q,q,KCDCU fqBOQT
SV&CW:xSVwZv'DG'NSPypDhKStKzC -$hslxIVzoivnp ,ethA:NCCGoi
tN!ljjP3fwJMwNelgUzzPGJlgihJ!d?q.d
pSPYgCuCJrIFtb
jQXg
pA.P LP,SPJi
DBcuBM:CixjJ$Jzkq,OLf3KLQLMGph$O 3DfiPHnXKuHMlyjxEiyZib3FaHV-oJa!zoc'XSP :CKGUhd?lgCOF$;;DTHZMlvvcmZAm;:iv'MMgO&Ywbc;BLCUd&vZINLIzkuTGZa
D.?


## The mathematical trick in self-attention

In [16]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [17]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [18]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [19]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [20]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


False

In [21]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [22]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [23]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [24]:
k.var()

tensor(1.0449)

In [25]:
q.var()

tensor(1.0700)

In [26]:
wei.var()

tensor(1.0918)

In [27]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [28]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [29]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [30]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [31]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

In [32]:
# French to English translation example:

# <--------- ENCODE ------------------><--------------- DECODE ----------------->
# les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>



### Full finished code, for reference

You may want to refer directly to the git repo instead though.

In [36]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 #一個批次并行訓練多少組tokens
block_size = 32 #一組裏面有多少tokens
max_iters = 1000 #訓練iteration多少次
eval_interval = 100 #多久做一次eval
learning_rate = 1e-3 #LR嘛 學習率
device = 'cuda' if torch.cuda.is_available() else 'cpu' #能用cuda用cuda
eval_iters = 200 #？ #算loss的時候采樣sample size
n_embd = 256 #表徵空間大小
n_head = 8 #multi-head attention
n_layer = 4 #訓練幾層block
dropout = 0.0 #
USE_RMSNORM = True #用Layer还是RMS

# ------------

torch.manual_seed(1337) #for replication


with open('input.txt', 'r', encoding='utf-8') as f: #讀一讀
    text = f.read()

#
chars = sorted(list(set(text))) #整理出所有unique character
vocab_size = len(chars) #得到總長度
#
stoi = { ch:i for i,ch in enumerate(chars) } #char to i 映射
itos = { i:ch for i,ch in enumerate(chars) } #i to char 映射
encode = lambda s: [stoi[c] for c in s] # str to list[int]
decode = lambda l: ''.join([itos[i] for i in l]) # list[int] to str

#
data = torch.tensor(encode(text), dtype=torch.long) #list[int] to tensor
n = int(0.9*len(data)) # 得到長度
train_data = data[:n] #劃分訓練集
val_data = data[n:] #劃分驗證集


# class RMSNorm(nn.Module):
#     def __init__(self, dim, eps=1e-8):
#         super().__init__()
#         self.eps = eps
#         self.gamma = nn.Parameter(torch.ones(dim))

#     def forward(self, x):
#         # x: [..., dim]
#         # 1) 算最后一维的均方
#         ms = (x * x).mean(-1, keepdim=True)
#         # 2) 开方加 eps
#         rms = torch.sqrt(ms + self.eps) #防止除以0 就废了
#         # 3) 归一化后乘 gamma
#         return self.gamma * (x / rms)
# 注释掉了 因为这个不是最快的方法。

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return F.rms_norm(x, (x.shape[-1],), weight=self.gamma, eps=self.eps)

#这个才是torch官方给的RMSNorm

Norm = RMSNorm if USE_RMSNORM else (lambda d: nn.LayerNorm(d))
# x²       = [1, 4, 9, 16, 25]
# ms       = (1+4+9+16+25)/5 = 55/5 = 11
# rms      = sqrt(11 + eps) ≈ 3.3166
# x / rms  = [0.3015, 0.6030, 0.9045, 1.2060, 1.5076]
# y        = gamma * 上式 = 同上（gamma=1）
# 其实你类比一下 这边算一次每个东西的平方+加起来平均+sqrt+广播除掉
# ayernorm是计算mean+计算var 这边只计算一次mean 所以少了一个reduction


#
def get_batch(split):
    # 根據split是啥進行操作
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    #torch.randint的語法,前者是最大值 至少抽出來之後能往後走block_size個；
    #(batch_size，)就是輸出的數據長什麽樣子啦 抽出來batch_size組扔進去
    x = torch.stack([data[i:i+block_size] for i in ix])
    #去data這個list[int]裏面把i到i+blocksize-1這些都拿出來

    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    #去data這個list[int]裏面把i+1到i+blocksize這些都拿出來

    #stack就是曡羅漢 把抓出來的batch_size個長度爲block_size的tensor堆叠
    #變成[B,T]
    x, y = x.to(device), y.to(device)
    return x, y #兩個[B,T]

@torch.no_grad() #套一層不計算梯度的裝飾器
def estimate_loss():
    out = {}
    model.eval() #切換評估模式
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters) #造200個格子
        for k in range(eval_iters): #循環200次
            X, Y = get_batch(split) #隨機抽一個
    #搞到手測試需要的兩個list[int]
            logits, loss = model(X, Y) #算loss
    #model = BigramLanguageModel()
    #在Pytorch的nn.Module裏，調用這個實例就是調用forward
            losses[k] = loss.item() #把loss吐出來
        out[split] = losses.mean()
        #字典 形成train和val上訓練loss的鍵值對
        #{'train': tensor(1.66), 'val': tensor(1.82)}形如這樣
    model.train() #切換回訓練模式
    return out #把字典吐出來

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__() #繼承nnModule
        self.key = nn.Linear(n_embd, head_size, bias=False)
        #[n_embd, head_size]的矩陣綫性層
#我們投喂的x 其實這個時候channel的C已經是表徵空間了
#但我們不打算直接用它 而是每個head都學習一套這個表徵的某一個側面的qkv
#然後把channel原來的n_embd長度改成head_size
        self.query = nn.Linear(n_embd, head_size, bias=False)
        #同上
        self.value = nn.Linear(n_embd, head_size, bias=False)
        #同上
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        #製造一個[T,T]大小的下三矩陣
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x) #[B,T,H]
        q = self.query(x) #[B,T,H]

        wei = q @ k.transpose(-2,-1) * C**-0.5
        #[B,T,H]@[B,H,T] = [B,T,T]
        #這樣就真的是每個token對每個token的打分了
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        #masked_fill做掩碼 把滿足要求的都改成負無窮
        wei = F.softmax(wei, dim=-1) #做softmax
        wei = self.dropout(wei) ## 注意力 dropout
        v = self.value(x) #把[B,T,H]提取
        out = wei @ v #[B,T,T] @ [B,T,H] = [B,T,H]
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):#自定義需要head數量+head長度
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        #全部放在一個list裏面一步一步來 注冊子模塊 追蹤參數
        self.proj = nn.Linear(n_embd, n_embd) # 1！！！
        #反正再映射一層
        self.dropout = nn.Dropout(dropout)
        #dropout一手

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        #每次從list裏面取出一個函數算一遍 輸出一個[B,T,H]*N就是[B,T,H*N]
        #不過在這份代碼裏面H*N=C是hardcode在裏面了 1！！！那個地方
        out = self.dropout(self.proj(out))
        return out #反正吐出來的是[B,T,C]

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),#放大四倍
            nn.ReLU(),#非綫性層
            nn.Linear(4 * n_embd, n_embd),#恢復原先參數量級
            nn.Dropout(dropout),#drop一手
        ) #反正sequential就是組裝一些基本block

    def forward(self, x):
        return self.net(x) #again Module的定義裏面直接調用feedforward

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):#需要C和N
        super().__init__()
        head_size = n_embd // n_head #這裏是這樣定義的 但也不一定
        self.sa = MultiHeadAttention(n_head, head_size) #寫過了
        self.ffwd = FeedFoward(n_embd) #寫過了
        self.n1 = Norm(n_embd) #寫過了
        self.n2 = Norm(n_embd) #寫過了

    def forward(self, x):
        x = x + self.sa(self.n1(x))  #layernorm+resnet處理梯度
        x = x + self.ffwd(self.n2(x)) #layernorm+resnet處理梯度
        return x             #ln + sa +ln + ffwd

class BigramLanguageModel(nn.Module): #好的我們重寫這個大類

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) #每個詞的表徵空間
        self.position_embedding_table = nn.Embedding(block_size, n_embd)#每個詞的位置編碼
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        #跑n_layer輪的Block 然後用*解開list放到sequential裏
        self.ln_f = Norm(n_embd) #做一層ln
        self.lm_head = nn.Linear(n_embd, vocab_size) #把每個token的維度表示映射成vocab size上的打分

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) #查表 把每個字符的對應的表徵空間
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))#對應的每個位置（第0，1，2，14這種）都有自己的nembd
        x = tok_emb + pos_emb #加起來啦 broadcast
        x = self.blocks(x) #跑一遍blocks
        x = self.ln_f(x) #跑一邊ln
        logits = self.lm_head(x) #把每個token的維度表示映射成vocab size上的打分

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) #展開 攤平batch 每個位置的logits
            targets = targets.view(B*T) #打扁平了
            loss = F.cross_entropy(logits, targets) #經典交叉熵搭配softmax計算loss

        return logits, loss #把logits吐出來 loss放出來

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            #
            idx_cond = idx[:, -block_size:] #取最後blocksize個token啦
            #
            logits, loss = self(idx_cond) #拿最後32個跑forward
            #
            logits = logits[:, -1, :] #只拿最後一組logits
            #
            probs = F.softmax(logits, dim=-1) # 走softmax抽取
            #
            idx_next = torch.multinomial(probs, num_samples=1) #按照概率抽一個
            #
            idx = torch.cat((idx, idx_next), dim=1) #拼接進去新的内容
        return idx

model = BigramLanguageModel() #instantiation
m = model.to(device) #把模型搬運到指定的設備
#
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters') #告訴我們有多少個參數

import time #引入计时器

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate) #這個是優化器

# warm-up：丢几步不算，等 GPU 初始化完成
for _ in range(3):
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

if device == 'cuda':
    torch.cuda.synchronize()
train_start = time.perf_counter() #开始计时

for iter in range(max_iters):

    #
    if iter % eval_interval == 0 or iter == max_iters - 1: #訓練中旬and結束前
        losses = estimate_loss() #估計一下哈
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    #
    xb, yb = get_batch('train') #根據訓練集 訓練：）

    #
    logits, loss = model(xb, yb) #喂到model裏面算logits和loss
    optimizer.zero_grad(set_to_none=True) #梯度清零
    loss.backward() #backprop
    optimizer.step() #更新權重

if device == 'cuda':
    torch.cuda.synchronize() #等所有GPU干完活才取时间戳
train_end = time.perf_counter() #结束咯

total_time = train_end - train_start #按照秒计时
print(f"\n[Speed] total train time: {total_time:.2f} s") #训练时间（秒）
print(f"[Speed] avg per iter:     {total_time / max_iters * 1000:.2f} ms") #每个循环耗时毫秒
print(f"[Speed] iters/sec:        {max_iters / total_time:.2f}") #每秒跑多少步

#
context = torch.zeros((1, 1), dtype=torch.long, device=device) #我啥也不説！
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist())) #從換行符開始生成兩千token 然後把第0條（本來也只有1條）拿出來
#然後再轉成list方便decode改回string


3.195713 M parameters
step 0: train loss 3.3470, val loss 3.3682
step 100: train loss 2.4452, val loss 2.4620
step 200: train loss 2.2650, val loss 2.2791
step 300: train loss 2.1346, val loss 2.1915
step 400: train loss 2.0658, val loss 2.1420
step 500: train loss 1.9981, val loss 2.0996
step 600: train loss 1.9470, val loss 2.0365
step 700: train loss 1.9100, val loss 2.0331
step 800: train loss 1.8572, val loss 1.9906
step 900: train loss 1.8214, val loss 1.9479
step 999: train loss 1.7987, val loss 1.9466

[Speed] total train time: 132.64 s
[Speed] avg per iter:     132.64 ms
[Speed] iters/sec:        7.54


Clown:
Rech cown, humself be madise a way to take Ond my called
My Londicy I will hert?

SeXEND:
A, away, my feand's wizenough outs, to time calt my well
Whoues is ees, I in latise in ov the count lings on transe. And no pear.
-huch on him speappiniss hiw you:
My furnorforess and whomen I seem that
And in on her piings to they sun, Tan his nust so most-for underand thrupter so;

In [ ]:
# [Speed] total train time: 128.23 s
# [Speed] avg per iter:     128.23 ms
# [Speed] iters/sec:        7.80
# 这个是layernorm

In [33]:
# 这个是RMSNorm

# [Speed] total train time: 132.64 s
# [Speed] avg per iter:     132.64 ms
# [Speed] iters/sec:        7.54

#小模型 看不出来差别 误差范围之内吧